# Actividad 1 – Dibujo de objetos 3D, visualización y proyección
## Creeper de Minecraft con PyOpenGL

Este notebook documenta y organiza el código de la actividad.
El programa completo se encuentra en `creeper_opengl.py`.


## 1. Instalación de dependencias


In [ ]:
# Ejecutar una sola vez
# !pip install PyOpenGL PyOpenGL_accelerate
# En Linux también puede ser necesario:
# !sudo apt-get install freeglut3-dev

## 2. Importaciones y constantes de color


In [ ]:
from OpenGL.GL   import *
from OpenGL.GLU  import *
from OpenGL.GLUT import *
import math, sys

# Paleta de colores del Creeper
GREEN_LIGHT  = (0.18, 0.70, 0.18)   # verde claro cuerpo
GREEN_DARK   = (0.08, 0.42, 0.08)   # verde oscuro detalle
BLACK_FACE   = (0.05, 0.05, 0.05)   # negro cara
GREEN_MID    = (0.12, 0.55, 0.12)   # verde medio pies/cuerpo

WIN_W, WIN_H = 1000, 800
cam_angle_x = 20.0
cam_angle_y = -40.0
cam_zoom    = 40.0
last_x = last_y = 0
mouse_btn = None

## 3. Primitiva: cubo unitario (`color_cube`)

Función base proporcionada en el enunciado, que dibuja un cubo de lado 1
centrado en el origen con todas las caras del mismo color.


In [ ]:
def color_cube(color):
    """Cubo unitario centrado en el origen con color uniforme."""
    r, g, b = color
    glColor3f(r, g, b)
    faces = [
        ([0,0,1],  [(-0.5,-0.5,0.5),(0.5,-0.5,0.5),(0.5,0.5,0.5),(-0.5,0.5,0.5)]),
        ([0,0,-1], [(0.5,-0.5,-0.5),(-0.5,-0.5,-0.5),(-0.5,0.5,-0.5),(0.5,0.5,-0.5)]),
        ([0,1,0],  [(-0.5,0.5,-0.5),(0.5,0.5,-0.5),(0.5,0.5,0.5),(-0.5,0.5,0.5)]),
        ([0,-1,0], [(-0.5,-0.5,0.5),(0.5,-0.5,0.5),(0.5,-0.5,-0.5),(-0.5,-0.5,-0.5)]),
        ([1,0,0],  [(0.5,-0.5,0.5),(0.5,-0.5,-0.5),(0.5,0.5,-0.5),(0.5,0.5,0.5)]),
        ([-1,0,0], [(-0.5,-0.5,-0.5),(-0.5,-0.5,0.5),(-0.5,0.5,0.5),(-0.5,0.5,-0.5)]),
    ]
    glBegin(GL_QUADS)
    for normal, verts in faces:
        glNormal3fv(normal)
        for v in verts:
            glVertex3fv(v)
    glEnd()
    # Aristas
    glColor3f(0.0, 0.0, 0.0)
    glBegin(GL_LINE_LOOP)
    for v in [(-0.5,-0.5,0.5),(0.5,-0.5,0.5),(0.5,0.5,0.5),(-0.5,0.5,0.5)]: glVertex3fv(v)
    glEnd()
    glBegin(GL_LINE_LOOP)
    for v in [(0.5,-0.5,-0.5),(-0.5,-0.5,-0.5),(-0.5,0.5,-0.5),(0.5,0.5,-0.5)]: glVertex3fv(v)
    glEnd()
    glBegin(GL_LINES)
    for a,b in [((-0.5,-0.5,0.5),(-0.5,-0.5,-0.5)),((0.5,-0.5,0.5),(0.5,-0.5,-0.5)),
                ((0.5,0.5,0.5),(0.5,0.5,-0.5)),((-0.5,0.5,0.5),(-0.5,0.5,-0.5))]:
        glVertex3fv(a); glVertex3fv(b)
    glEnd()

## 4. Función auxiliar: ortoedro de cubos (`draw_orthohedron`)

Permite construir cualquier bloque rectangular de cubos unitarios.
Con `hollow=True` sólo se dibujan los cubos de la capa exterior,
maximizando los elementos huecos tal como exige el enunciado.


In [ ]:
def draw_orthohedron(cx, cy, cz, W, D, H, color, hollow=True):
    """
    Bloque W×D×H de cubos unitarios centrado en (cx, cy, cz).
    hollow=True → sólo capa exterior (elementos huecos).
    """
    ox = cx - W/2.0 + 0.5
    oy = cy - H/2.0 + 0.5
    oz = cz - D/2.0 + 0.5
    for ix in range(W):
        for iy in range(H):
            for iz in range(D):
                if hollow:
                    if not (ix==0 or ix==W-1 or iy==0 or iy==H-1 or iz==0 or iz==D-1):
                        continue
                glPushMatrix()
                glTranslatef(ox+ix, oy+iy, oz+iz)
                color_cube(color)
                glPopMatrix()

## 5. Elementos de la cara del Creeper


In [ ]:
def draw_face(head_cx, head_cy, head_cz, head_H):
    """Dibuja ojos y boca negros sobre la cara frontal de la cabeza."""
    z_face = head_cz + 4.0 + 0.01
    eye_positions = [
        (-2, head_cy+1.5), (-1, head_cy+1.5),
        (-2, head_cy+0.5), (-1, head_cy+0.5),
        ( 1, head_cy+1.5), ( 2, head_cy+1.5),
        ( 1, head_cy+0.5), ( 2, head_cy+0.5),
    ]
    mouth_positions = [
        (-1, head_cy-0.5),(0, head_cy-0.5),(1, head_cy-0.5),
        (-2, head_cy-1.5),(-1, head_cy-1.5),(0, head_cy-1.5),(1, head_cy-1.5),(2, head_cy-1.5),
        (-1, head_cy-2.5),(0, head_cy-2.5),(1, head_cy-2.5),
    ]
    for (x, y) in eye_positions + mouth_positions:
        glPushMatrix()
        glTranslatef(head_cx+x, y, z_face)
        glScalef(1.0, 1.0, 0.02)
        color_cube(BLACK_FACE)
        glPopMatrix()

## 6. Ensamblaje del Creeper completo

| Elemento | Ancho X | Fondo Z | Alto Y | Centro Y | Notas |
|----------|---------|---------|--------|----------|-------|
| Pie izq  | 8 | 4 | 6 | 3  | CentroX = -5 |
| Pie der  | 8 | 4 | 6 | 3  | CentroX = +5 |
| Cuerpo   | 8 | 4 | 12 | 11 | Solapa 1 cubo con pies |
| Cabeza   | 8 | 8 | 8 | 21 | Centrada con cuerpo |


In [ ]:
def draw_creeper():
    """Ensambla el Creeper completo."""
    # Pies
    draw_orthohedron(-5, 3.0, 0.0, 8, 4, 6, GREEN_MID,   hollow=True)
    draw_orthohedron(+5, 3.0, 0.0, 8, 4, 6, GREEN_DARK,  hollow=True)
    # Cuerpo (base en Y=5 → solapamiento de 1 cubo con pies)
    draw_orthohedron(0, 11.0, 0.0, 8, 4, 12, GREEN_LIGHT, hollow=True)
    # Cabeza (sobre el cuerpo, centrada)
    draw_orthohedron(0, 21.0, 0.0, 8, 8,  8, GREEN_LIGHT, hollow=True)
    # Cara
    draw_face(0, 21.0, 0.0, 8)

## 7. Proyecciones

Se implementan tres tipos de proyección:
- **Ortográfica** paralela (vistas frontal y lateral)
- **Perspectiva** cónica (vista orbital interactiva)
- **Oblicua de gabinete** (cizallamiento con factor 0.5 y ángulo 45°)


In [ ]:
def set_ortho_projection(W, H, extent=30):
    ratio = W/H if H>0 else 1
    glMatrixMode(GL_PROJECTION); glLoadIdentity()
    glOrtho(-extent*ratio, extent*ratio, -extent, extent, -200, 200)
    glMatrixMode(GL_MODELVIEW)

def set_perspective_projection(W, H):
    glMatrixMode(GL_PROJECTION); glLoadIdentity()
    gluPerspective(45.0, W/H if H>0 else 1, 0.1, 500.0)
    glMatrixMode(GL_MODELVIEW)

def set_oblique_cabinet_projection(W, H, extent=30):
    ratio = W/H if H>0 else 1
    glMatrixMode(GL_PROJECTION); glLoadIdentity()
    glOrtho(-extent*ratio, extent*ratio, -extent, extent, -200, 200)
    f, a = 0.5, math.radians(45)
    shear = [1,0,0,0, 0,1,0,0, -f*math.cos(a),-f*math.sin(a),1,0, 0,0,0,1]
    glMultMatrixf(shear)
    glMatrixMode(GL_MODELVIEW)

## 8. Bucle principal GLUT

Para ejecutar el programa completo con los 4 viewports, ejecuta desde terminal:

```bash
python creeper_opengl.py
```

**Controles:**
- `Drag izquierdo` en VP inferior-izquierdo → rotar vista orbital
- `Drag derecho` en VP inferior-izquierdo → zoom
- `ESC` o `q` → salir


In [ ]:
# Para ejecutar desde notebook (abre ventana GLUT):
# import subprocess
# subprocess.Popen(['python', 'creeper_opengl.py'])
print("Ejecuta:  python creeper_opengl.py")